# Clasificación de correos electrónicos spam/ham

## 1. Importar conjunto de datos

En primer lugar, importamos el conjunto de datos "spam.csv". 

La colección `SMS Spam` es un conjunto de datos que contiene 5.574 mensajes SMS en inglés con la etiqueta `Spam` o `Ham` (no spam). 


In [ ]:
import pandas as pd

data = pd.read_csv('https://raw.githubusercontent.com/mohitgupta-omg/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv', encoding='latin-1')

data.head()

In [ ]:
data.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1, inplace=True)

data.columns = ['label', 'text']

data.head()

## 2. Análisis exploratorio de datos (AED o EDA)

In [ ]:
data.isna().sum()

In [ ]:
data.shape

In [ ]:
data['label'].value_counts(normalize = True).plot.bar()

## 3. Preprocesamiento de textos

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
pattern = re.compile(r"[^a-zA-Z]+")

corpus = []

for doc in data["text"].astype(str):
    r = pattern.sub(" ", doc).lower()
    tokens = r.split()
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    corpus.append(" ".join(tokens))

data["text"] = corpus 
data.head()


## 4. División dataset: entrenamiento-prueba

In [ ]:
from sklearn.model_selection import train_test_split

X = data["text"].astype(str)
y = data["label"] 

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, # 20 % de los datos van a test > resto a train
    random_state=42,
    stratify=y # no cambia el porcentaje, solo asegura que la proporción de clases sea la misma en train y test
)
X_train.shape, X_test.shape

## 5. Extracción de características y entrenamiento

Aquí utilizamos el modelo de bolsa de palabras `TfidVectorizer` para convertir el texto depurado en características numéricas. Esto es necesario para entrenar el modelo de aprendizaje automático.

👉 Propongo `TfidfVectorizer` en lugar de `CountVectorizer` porque:
- pondera menos las palabras muy frecuentes (por ejemplo, CountVectorizer da mucho peso a "the" simplemente porque aparece mucho, no porque sea más informativa).
- da más peso a palabras discriminativas
- suele producir mejor rendimiento en clasificación de textos
- es el baseline estándar moderno en PLN clásico

CountVectorizer es más simple, pero demasiado ingenuo para la mayoría de tareas reales.

Con `TfidVectorizer`:
- TF: cuán importante es la palabra en el documento
- IDF: cuán rara es la palabra en el corpus


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1,2),      # uni+bi-gramas: mejora mucho en clase
        min_df=2
    )),
    ("model", LinearSVC())
])

clf.fit(X_train, y_train)


## 6. Evaluación del modelo

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred = clf.predict(X_test) # aplica todo el pipeline anterior y devuelve la clase predicha para cada documento de test

print(classification_report(y_test, y_pred)) # Esto imprime una tabla con métricas estándar de clasificación (precision, recall, f1-score, support) para cada clase

cm = confusion_matrix(y_test, y_pred) # Matriz de confusión: filas=clase real, columnas=clase predicha
disp = ConfusionMatrixDisplay(confusion_matrix=cm) # Convierte la matriz en un gráfica --> cmap="Blues"
disp.plot(cmap="Blues", values_format="d") # escala de color / values_format="d" → números enteros
plt.show()


## 7. Probemos el modelo entrenado

In [ ]:
print(clf.classes_)

In [ ]:
# ------------- PROBAR TEXTO NUEVO ----------------

# Texto nuevo que quieres clasificar
texto_nuevo1 = """
Hey, are we still meeting tomorrow afternoon? Let me know what time works best for you. See you!
"""

# Predecir la clase
pred = clf.predict([texto_nuevo1])[0]

# Mostrar la predicción
print("Etiqueta predicha:", pred)

In [ ]:
# ------------- PROBAR TEXTO NUEVO ----------------

# Texto nuevo que quieres clasificar
texto_nuevo2 = """
Congratulations! You have been selected to receive a $1000 gift card.
Reply NOW to claim your prize before the offer expires!
"""

# Predecir la clase
pred = clf.predict([texto_nuevo2])[0]

# Mostrar la predicción
print("Etiqueta predicha:", pred)

## Opcional: Analizar por qué ha etiquetado así el nuevo texto 2 (spam)

### 1. Preparar elementos del pipeline

In [ ]:
import numpy as np

# Texto ya clasificado
texto = texto_nuevo2

# Acceder a partes del pipeline
tfidf = clf.named_steps["tfidf"]
svm = clf.named_steps["model"]


### 2. Vectorizar el texto y obtener features

In [ ]:
X_vec = tfidf.transform([texto])              # (1, n_features)
feature_names = np.array(tfidf.get_feature_names_out())


### 3. Identificar la clase que nos interesa

In [ ]:
print("Clases:", clf.classes_)

Aquí hay que indicar dentro de `index` la clase que nos interesa:

In [ ]:
interest_class_index = list(clf.classes_).index('spam')

### 4. Calcular contribuciones palabra (clase de interés)

Ojo: 

En datasets con clases binarias (0/1, ham/spam, etc.):
✔️ SIEMPRE se usa svm.coef_[0]

En datasets con multiclase (World/Sports/Business/Tech, por ejemplo):
✔️ Debes usar svm.coef_[indice_de_la_clase]


In [ ]:
# Pesos del modelo para la clase que nos interesa
weights = svm.coef_[0]

# Vector TF-IDF del texto
tfidf_values = X_vec.toarray()[0]

# Contribución = tfidf * peso
contributions = tfidf_values * weights


### 5. Mostrar las palabras más influyentes

In [ ]:
top_idx = np.argsort(contributions)[-15:][::-1]

print("Palabras / n-gramas que más contribuyen a la clase de interés:\n")

for idx in top_idx:
    if tfidf_values[idx] > 0:
        print(f"{feature_names[idx]:<20} "
              f"tfidf={tfidf_values[idx]:.3f} "
              f"peso={weights[idx]:.3f} "
              f"contrib={contributions[idx]:.3f}")
